## Customer Order Web Form
A simple Flask‑based example that shows how to build a web form for customer orders.  
The notebook creates an app with routes for rendering the form (`/`), processing the POSTed data (`/submit`), validating required fields and the price, and inserting a new `Customer` record into the database via SQLAlchemy.  
After a successful submission the user is redirected to a success page. This cell documents the workflow for running the app and testing the form.

In [ ]:
from flask import render_template, request, redirect, url_for, flash
from app import create_app, db
from models import Customer

# Create the Flask app (tables are created in create_app)
app = create_app()

# ---------------------
# ROUTES MUST BE TOP-LEVEL
# ---------------------

@app.route("/")
def index():
    return render_template("form.html")  # make sure templates/form.html exists


@app.route("/submit", methods=["POST"])
def submit():
    firstname = (request.form.get("firstName") or "").strip()
    lastname  = (request.form.get("lastName") or "").strip()
    email     = (request.form.get("email") or "").strip()
    phone     = (request.form.get("phone") or "").strip()
    category  = (request.form.get("category") or "").strip()
    product   = (request.form.get("product") or "").strip()
    price_raw = (request.form.get("price") or "").strip()

    if not firstname or not lastname or not product or not category or not price_raw:
        flash("First name, Last name, Product, Category, and Price are required.", "error")
        return redirect(url_for("index"))

    try:
        price = float(price_raw)
    except ValueError:
        flash("Price must be a real number.", "error")
        return redirect(url_for("index"))

    if price < 0:
        flash("Price must be ≥ 0.", "error")
        return redirect(url_for("index"))

    full_name = f"{firstname} {lastname}"

    # ORM INSERT
    with app.app_context():
        customer = Customer(
            first_name=firstname,
            last_name=lastname,
            email=email or None,
            phone=phone or None,
            product=product,
            category=category,
            price=price,
        )
        db.session.add(customer)
        db.session.commit()

    return redirect(url_for("success"))


@app.route("/success")
def success():
    return "Form submitted successfully! Data saved to the database."


if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)